# Exercise 5.4

> After saving the weights, load the model and optimizer in a new Python session or Jupyter notebook file and continue pretraining it for one more epoch using the `train_model_simple` function.


## Load the saved training state


In [ ]:
from pathlib import Path

import requests
import tiktoken
import torch

from CH04_Implementing_a_GPT_model_from_scratch_to_generate_text.gpt import (
    GPTModel,
    create_dataloader_v1,
)
from CH05_Pretraining_on_Unlabeled_Data.gpt_train import train_model_simple

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

tokenizer = tiktoken.get_encoding("gpt2")
checkpoint_path = Path("model_and_optimizer.pth")
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

saved_context_length = checkpoint["model_state_dict"]["pos_emb.weight"].shape[0]
assert saved_context_length == GPT_CONFIG_124M["context_length"]

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
del checkpoint
model.train()

print(f"Continuing training on: {device}")

## Recreate the training and validation loaders


In [ ]:
data_path = Path("the-verdict.txt")
data_url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
    "main/ch02/01_main-chapter-code/the-verdict.txt"
)

if not data_path.exists():
    response = requests.get(data_url, timeout=30)
    response.raise_for_status()
    data_path.write_text(response.text, encoding="utf-8")

text_data = data_path.read_text(encoding="utf-8")
split_idx = int(0.90 * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

torch.manual_seed(123)
context_length = GPT_CONFIG_124M["context_length"]

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=context_length,
    stride=context_length,
    drop_last=True,
    shuffle=True,
    num_workers=0,
)
val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=context_length,
    stride=context_length,
    drop_last=False,
    shuffle=False,
    num_workers=0,
)

print(f"Batches: train={len(train_loader)}, validation={len(val_loader)}")

## Continue pretraining for one epoch


In [ ]:
train_losses, val_losses, tokens_seen = train_model_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs=1,
    eval_freq=5,
    eval_iter=5,
    start_context="Every effort moves you",
    tokenizer=tokenizer,
)